# House Price Prediction
### Kaggle Competition: House Prices - Advanced Regression Techniques

---

## Overview
In this notebook, we will predict house sale prices using the Ames Housing dataset.
We apply end-to-end machine learning — from EDA to a production-ready Pipeline.

## Workflow
1. Data Loading
2. Exploratory Data Analysis (EDA)
3. Data Preprocessing & Feature Engineering
4. Pipeline Construction
5. Model Training & Evaluation
6. Test Prediction & Submission

## Result
| Model | Validation R2 | Kaggle Score (RMSLE) |
|-------|--------------|----------------------|
| Ridge Regression (Pipeline) | 0.9057 | 0.13538 |

## 1. Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OrdinalEncoder, OneHotEncoder
from sklearn.linear_model import Ridge
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error

import warnings
warnings.filterwarnings('ignore')

## 2. Load Data

In [ ]:
train = pd.read_csv('/kaggle/input/competitions/house-prices-advanced-regression-techniques/train.csv')
test  = pd.read_csv('/kaggle/input/competitions/house-prices-advanced-regression-techniques/test.csv')

# Save test IDs for submission
test_ids = test['Id'].copy()

print(f'Train shape : {train.shape}')
print(f'Test shape  : {test.shape}')
train.head()

## 3. Exploratory Data Analysis (EDA)

In [ ]:
# Target Variable: SalePrice Distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(train['SalePrice'], kde=True, ax=axes[0], color='steelblue')
axes[0].set_title('SalePrice - Before Log Transform (Right Skewed)', fontsize=12)
axes[0].set_xlabel('SalePrice')

sns.histplot(np.log1p(train['SalePrice']), kde=True, ax=axes[1], color='seagreen')
axes[1].set_title('SalePrice - After Log Transform (Normal)', fontsize=12)
axes[1].set_xlabel('log1p(SalePrice)')

plt.suptitle('Log Transformation Fixes Right Skew', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print(f'Skewness before log : {train["SalePrice"].skew():.4f}')
print(f'Skewness after log  : {np.log1p(train["SalePrice"]).skew():.4f}')

In [ ]:
# Missing Values
missing = train.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)

plt.figure(figsize=(12, 5))
missing.plot(kind='bar', color='coral')
plt.title('Missing Values per Column', fontsize=13)
plt.ylabel('Missing Count')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

print(f'Total columns with missing values: {len(missing)}')
print(missing)

In [ ]:
# Correlation with SalePrice
num_cols = train.select_dtypes(include=['int64', 'float64']).columns.tolist()
corr = train[num_cols].corr()['SalePrice'].sort_values(ascending=False)

print('Top 10 features correlated with SalePrice:')
print(corr.head(10))

top_cols = corr.head(10).index.tolist()
plt.figure(figsize=(10, 8))
sns.heatmap(train[top_cols].corr(), annot=True, fmt='.2f',
            cmap='coolwarm', square=True)
plt.title('Correlation Heatmap - Top 10 Features', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Top Features vs SalePrice
top_features = ['OverallQual', 'GrLivArea', 'GarageCars',
                'TotalBsmtSF', 'FullBath', 'YearBuilt']

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

for i, col in enumerate(top_features):
    axes[i].scatter(train[col], train['SalePrice'], alpha=0.4, color='steelblue')
    axes[i].set_xlabel(col)
    axes[i].set_ylabel('SalePrice')
    axes[i].set_title(f'{col} vs SalePrice')

plt.suptitle('Top Features vs SalePrice', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# OverallQual Box Plot
plt.figure(figsize=(10, 5))
sns.boxplot(x='OverallQual', y='SalePrice', data=train, palette='coolwarm')
plt.title('Overall Quality vs Sale Price', fontsize=13)
plt.tight_layout()
plt.show()

## 4. Data Preprocessing & Feature Engineering

In [ ]:
def preprocess(data):
    data = data.copy()

    # Handle Missing Values
    # These columns: NaN means the feature does not exist — fill with 'None'
    none_cols = ['PoolQC', 'MiscFeature', 'Alley', 'Fence', 'FireplaceQu',
                 'GarageType', 'GarageFinish', 'GarageQual', 'GarageCond',
                 'BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1',
                 'BsmtFinType2', 'MasVnrType']
    for col in none_cols:
        if col in data.columns:
            data[col] = data[col].fillna('None')

    # Numerical — fill with median or 0
    data['LotFrontage'] = data['LotFrontage'].fillna(data['LotFrontage'].median())
    data['MasVnrArea']  = data['MasVnrArea'].fillna(0)
    data['GarageYrBlt'] = data['GarageYrBlt'].fillna(0)

    # Remaining missing values
    for col in data.select_dtypes(include='float64').columns:
        data[col] = data[col].fillna(data[col].median())
    for col in data.select_dtypes(include='object').columns:
        data[col] = data[col].fillna(data[col].mode()[0])

    # Feature Engineering
    # Total bathrooms
    data['TotalBath']   = (data['FullBath'] + data['HalfBath'] +
                           data['BsmtFullBath'] + data['BsmtHalfBath'])
    # Total square footage
    data['TotalSF']     = data['TotalBsmtSF'] + data['1stFlrSF'] + data['2ndFlrSF']
    # Age of house at time of sale
    data['HouseAge']    = data['YrSold'] - data['YearBuilt']
    # Whether the house was remodeled
    data['IsRemodeled'] = (data['YearRemodAdd'] != data['YearBuilt']).astype(int)

    return data

train = preprocess(train)
test  = preprocess(test)

print('Preprocessing complete!')
print(f'New features added : TotalBath, TotalSF, HouseAge, IsRemodeled')
print(f'Missing in train   : {train.isnull().sum().sum()}')
print(f'Missing in test    : {test.isnull().sum().sum()}')

## 5. Pipeline Construction

In [ ]:
# Separate target and features
y     = np.log1p(train['SalePrice'])
train = train.drop(['SalePrice', 'Id'], axis=1)
test  = test.drop(['Id'], axis=1)

# Define column types
ordinal_cols = ['ExterQual', 'ExterCond', 'BsmtQual', 'BsmtCond', 'HeatingQC',
                'KitchenQual', 'FireplaceQu', 'GarageQual', 'GarageCond', 'PoolQC']

nominal_cols  = [col for col in train.select_dtypes(include='object').columns
                 if col not in ordinal_cols]

numerical_cols = train.select_dtypes(include=['int64', 'float64']).columns.tolist()

print(f'Numerical features : {len(numerical_cols)}')
print(f'Ordinal features   : {len(ordinal_cols)}')
print(f'Nominal features   : {len(nominal_cols)}')

# Quality category order
quality_cats = ['None', 'Po', 'Fa', 'TA', 'Gd', 'Ex']

# ColumnTransformer
preprocessor = ColumnTransformer(transformers=[
    ('num', StandardScaler(), numerical_cols),
    ('ord', OrdinalEncoder(
                categories=[quality_cats] * len(ordinal_cols),
                handle_unknown='use_encoded_value',
                unknown_value=-1), ordinal_cols),
    ('nom', OneHotEncoder(handle_unknown='ignore', sparse_output=False), nominal_cols)
])

# Full Pipeline
pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', Ridge(alpha=10))
])

print('\nPipeline created successfully!')
print(pipeline)

## 6. Model Training & Evaluation

In [ ]:
# Train-Validation Split
X_train, X_val, y_train, y_val = train_test_split(
    train, y, test_size=0.2, random_state=42)

# Fit pipeline
pipeline.fit(X_train, y_train)

# Evaluate
val_pred = pipeline.predict(X_val)
r2   = r2_score(y_val, val_pred)
rmse = np.sqrt(mean_squared_error(y_val, val_pred))

print('=' * 40)
print('       Model Evaluation Results')
print('=' * 40)
print(f'  R2 Score : {r2:.4f}')
print(f'  RMSE     : {rmse:.4f}')
print('=' * 40)

# Actual vs Predicted Plot
plt.figure(figsize=(8, 6))
plt.scatter(y_val, val_pred, alpha=0.4, color='steelblue')
plt.plot([y_val.min(), y_val.max()],
         [y_val.min(), y_val.max()],
         'r--', linewidth=2, label='Perfect Prediction')
plt.xlabel('Actual log1p(SalePrice)')
plt.ylabel('Predicted log1p(SalePrice)')
plt.title(f'Actual vs Predicted  |  R2 = {r2:.4f}', fontsize=13)
plt.legend()
plt.tight_layout()
plt.show()

## 7. Test Prediction & Submission

In [ ]:
# Retrain on full training data for best performance
pipeline.fit(train, y)

# Predict on test set
test_pred    = pipeline.predict(test)
final_prices = np.expm1(test_pred)

print('Test Prediction Statistics:')
print(f'  Mean price : ${final_prices.mean():>12,.0f}')
print(f'  Min price  : ${final_prices.min():>12,.0f}')
print(f'  Max price  : ${final_prices.max():>12,.0f}')

# Save submission file
submission = pd.DataFrame({'Id': test_ids, 'SalePrice': final_prices})
submission.to_csv('/kaggle/working/submission.csv', index=False)

print('\nSubmission file saved!')
print(f'Shape: {submission.shape}')
submission.head(10)

---
## Summary

### Steps Performed
- **EDA**: Distribution analysis, missing value visualization, correlation heatmap, scatter plots
- **Preprocessing**: Handled missing values using None / median / mode strategy
- **Feature Engineering**: Created TotalBath, TotalSF, HouseAge, IsRemodeled
- **Encoding**: OrdinalEncoder for quality columns, OneHotEncoder for nominal columns
- **Scaling**: StandardScaler for numerical features
- **Model**: Ridge Regression (alpha=10) wrapped inside a clean sklearn Pipeline

### Final Results
| Metric | Score |
|--------|-------|
| Validation R2 | 0.9057 |
| Kaggle Public RMSLE | 0.13538 |